# FSCT 8561 — Lab 3
## Building an OTP Authentication Server

**Course:** FSCT 8561 — Security Applications  
**Lab Duration:** 3 Hours

### Overview

In Lab 1, you built a **stateful TCP client–server application**.

In Lab 2, you used Python networking concepts to **discover exposed services**.

In Lab 3, you will add another important security concept:

> **Authentication — How does a server decide whether a user should be trusted?**

You will build a simple client–server authentication system that combines:

1. a username,
2. a password that is verified using a cryptographic hash, and
3. a Time-Based One-Time Password (**TOTP**).

The final workflow will be:

```text
Client                    Server
  |                          |
  |--- username/password --->|
  |                          | Verify password hash
  |<----- OTP_REQUIRED ------|
  |                          |
  |-------- OTP ------------>|
  |                          | Verify TOTP
  |<---- ACCESS_GRANTED -----|
```

This lab focuses on understanding authentication logic and secure credential handling. It is an educational prototype, not a production authentication system.

## Learning Objectives

By the end of this lab, you should be able to:

- explain the role of authentication in network security,
- explain why passwords should not be stored directly,
- hash and verify passwords using Python's `hashlib`,
- generate and verify TOTP values using `PyOTP`,
- integrate password and OTP verification into a stateful TCP server,
- build a client that participates in a multi-step authentication protocol,
- handle common authentication failures without crashing,
- analyze the benefits and limitations of password + TOTP authentication.

## Prerequisites

You should have completed **Lab 1 and Lab 2**.

You should already be comfortable with:

- `socket.AF_INET` and `socket.SOCK_STREAM`
- `bind()`, `listen()`, `accept()`, and `connect()`
- `sendall()` and `recv()`
- `encode()` and `decode()`
- `while` loops and `if / elif / else`
- functions
- dictionaries
- `try / except`

We will reuse the stateful client–server structure from Lab 1 rather than introducing a completely new networking architecture.

## Required Reading & Tutorials

- *Mastering Python for Networking and Security* — Chapter 4
   -	https://learning.oreilly.com/library/view/mastering-python-for/9781839217166/
   -	Source code on: https://github.com/PacktPublishing/
- Python `hashlib` documentation : https://docs.python.org/3/library/hashlib.html
- PyOTP documentation : https://pyauth.github.io/pyotp/ 
- Python `getpass` documentation : https://docs.python.org/3/library/getpass.html


Install PyOTP before starting:

```bash
pip install pyotp
```

# Part 1 — Start with the Authentication Problem

Suppose a server stores this:

```python
users = {
    "alice": "mypassword123"
}
```

This is easy to implement, but it means the password is stored in **plaintext**.

If the user database is exposed, the password is immediately visible.

Instead of storing the password itself, we want the server to store a value derived from it:

```text
Password
   ↓
Hash function
   ↓
Password hash
```

During authentication:

```text
Password entered by user
          ↓
       Hash it
          ↓
Compare with stored hash
          ↓
      Match?
      /    \
    Yes     No
```

The server therefore does not need to compare the plaintext password directly.

# Part 2 — Learn `hashlib` One Step at a Time

Python includes the `hashlib` module for cryptographic hash functions.

For this lab, we will use SHA-256 to understand the mechanics of password hashing.

> **Important:** SHA-256 by itself is not the recommended password-storage mechanism for production systems. Real systems should use a password-hashing function such as Argon2, bcrypt, scrypt, or PBKDF2 with appropriate salts and parameters. We use SHA-256 here because the lab's goal is to understand the authentication workflow before introducing production password-storage design.

### Step 1 — Hash a String

Hash functions operate on **bytes**, so the password must first be encoded.

In [ ]:
import hashlib

password = "Cyber123!"

password_bytes = password.encode()

password_hash = hashlib.sha256(
    password_bytes
).hexdigest()

print("Password hash:")
print(password_hash)

Run the cell several times.

### Observe

- Does the same password produce the same hash?
- How long is the hexadecimal SHA-256 output?
- Can you look at the hash and determine the original password?

Now change one character in the password and run the code again.

### Step 2 — Turn Hashing into a Function

Instead of repeating the hashing code, create:

```python
def hash_password(password):
```

The function should accept a password and return its SHA-256 hexadecimal digest.

In [ ]:
import hashlib

def hash_password(password):

    return hashlib.sha256(
        password.encode()
    ).hexdigest()


print(hash_password("Cyber123!"))

### Step 3 — Verify a Password

The server can store the expected hash and compare it with the hash of the password entered by the user.

Complete the logic below.

In [ ]:
stored_hash = hash_password("Cyber123!")

entered_password = "Cyber123!"

entered_hash = hash_password(
    entered_password
)

if entered_hash == stored_hash:
    print("Password accepted")
else:
    print("Password rejected")

## Your Task

Create a function:

```python
def verify_password(password, stored_hash):
```

It should return:

```python
True
```

when the hashes match and:

```python
False
```

otherwise.

Test it with both a correct and an incorrect password.

# Part 3 — Create a Simple User Database

For this lab, use a small Python dictionary to represent the server's user database.

Each user needs:

- a password hash,
- later, a TOTP secret.

Start with the password hash only.

In [ ]:
users = {
    "alice": {
        "password_hash": hash_password(
            "Cyber123!"
        )
    }
}

print(users["alice"])

### Security Check

Your dictionary should **not** contain:

```text
Cyber123!
```

as the stored password.

The server should store only the derived password value.

Later in the lab, we will add an OTP secret to the same user record.

# Part 4 — Introduce Time-Based One-Time Passwords

A normal password is static: it may remain the same for weeks or months.

A **Time-Based One-Time Password (TOTP)** changes automatically over time.

Conceptually:

```text
Shared secret + Current time
          ↓
      TOTP algorithm
          ↓
     6-digit code
```

The client/user and server can independently calculate the expected code because they share the same secret and have approximately synchronized clocks.

A typical TOTP is valid only for a short time window.

This gives us two authentication factors in our educational system:

```text
Something the user KNOWS
        ↓
     Password

Something the user HAS
        ↓
TOTP secret / authenticator
```

# Part 5 — Generate a TOTP Secret with PyOTP

Import PyOTP:

In [ ]:
import pyotp

secret = pyotp.random_base32()

print("TOTP secret:")
print(secret)

Run this cell more than once.

### Observe

Does `pyotp.random_base32()` generate the same value every time?

Each account should have its own secret.

For this lab, the secret is displayed so you can configure and test your own account. In a real system, TOTP secrets must be protected carefully because anyone who obtains the secret can generate valid OTP values.

# Part 6 — Generate and Verify a TOTP

Create a TOTP object from the secret:

In [ ]:
totp = pyotp.TOTP(secret)

current_otp = totp.now()

print("Current OTP:", current_otp)

The value changes as the TOTP time window changes.

Now verify it:

In [ ]:
entered_otp = current_otp

if totp.verify(entered_otp):
    print("OTP accepted")
else:
    print("OTP rejected")

## Your Task

Experiment with:

1. the current OTP,
2. an incorrect six-digit value,
3. a previously generated OTP after it expires.

Record what you observe.

Then create:

```python
def verify_otp(secret, otp):
```

The function should return `True` or `False`.

In [ ]:
def verify_otp(secret, otp):

    totp = pyotp.TOTP(secret)

    return totp.verify(otp)

# Part 7 — Add the TOTP Secret to the User Account

Now combine the two pieces of account information:

```python
users = {
    "alice": {
        "password_hash": ...,
        "totp_secret": ...
    }
}
```

Generate one secret for your test user and keep that same secret during your testing session.

Do **not** generate a new secret every time the user tries to log in. If the secret changes, the server and authenticator will no longer calculate the same OTP.

In [ ]:
alice_secret = pyotp.random_base32()

users = {
    "alice": {
        "password_hash": hash_password(
            "Cyber123!"
        ),
        "totp_secret": alice_secret
    }
}

print(
    "Alice's TOTP secret:",
    users["alice"]["totp_secret"]
)

# Part 8 — Optional Provisioning URI

Authenticator applications commonly use a provisioning URI to configure a TOTP account.

PyOTP can generate one:

In [ ]:
totp = pyotp.TOTP(
    users["alice"]["totp_secret"]
)

uri = totp.provisioning_uri(
    name="alice",
    issuer_name="FSCT8561-Lab3"
)

print(uri)

You may use the provisioning information to configure your own authenticator for the lab.

### Think About It

Why should the provisioning URI and TOTP secret **not** be exposed publicly?

# Part 9 — Design the Authentication Protocol Before Coding

Before modifying the server, define how the client and server will communicate.

We will reuse the idea of an **application protocol** from Lab 1.

Use these messages:

```text
AUTH|username|password
OTP|123456
```

Possible server responses:

```text
OTP_REQUIRED
ACCESS_GRANTED
ACCESS_DENIED
ERROR|message
```

The authentication sequence is:

```text
CLIENT                              SERVER

AUTH|alice|password  -------------> Check username
                                      ↓
                                    Check password hash

OTP_REQUIRED          <------------ Password accepted

OTP|123456            ------------> Verify TOTP

ACCESS_GRANTED        <------------ OTP accepted
```

If the password is wrong:

```text
AUTH|alice|wrongpass  ------------>

ACCESS_DENIED         <------------
```

### Important

The server should verify the password **before** verifying the OTP.

This creates a clear, stateful authentication workflow and directly extends the protocol/state concepts from Lab 1.

# Part 10 — Build the Authentication Server

Create:

```text
auth_server.py
```

Reuse your Lab 1 TCP server structure:

1. create the socket,
2. bind to `127.0.0.1`,
3. listen,
4. accept a client,
5. receive messages,
6. interpret your authentication protocol,
7. send responses,
8. close the connection cleanly.

Start with one client at a time. Multi-client support is not required for the core Lab 3 implementation.

## Server Step A — Receive `AUTH`

The server should expect:

```text
AUTH|username|password
```

You can split the message:

```python
parts = message.split("|")
```

Validate the structure before using its fields.

The server should then:

1. check whether the username exists,
2. hash the submitted password,
3. compare it with the stored hash,
4. send `OTP_REQUIRED` only if the password is correct.

Do not allow the client to proceed to the OTP stage after a failed password.

## Server Step B — Maintain Authentication State

Your server needs to remember where the client is in the authentication process.

For example:

```python
password_verified = False
```

After successful password verification:

```python
password_verified = True
```

Then the server can accept an `OTP|...` message only when the password has already been verified.

This is another example of **stateful server behavior**, just like Lab 1.

## Server Step C — Receive and Verify `OTP`

When the server receives:

```text
OTP|123456
```

it should:

1. verify that the password stage was completed,
2. retrieve the correct user's TOTP secret,
3. verify the OTP with PyOTP,
4. send `ACCESS_GRANTED` or `ACCESS_DENIED`.

A client should **not** be able to send an OTP first and bypass the password stage.

# Part 11 — Build the Authentication Client

Create:

```text
auth_client.py
```

The client should:

1. connect to the authentication server,
2. ask for a username,
3. ask for a password,
4. send `AUTH|username|password`,
5. wait for the server response,
6. ask for an OTP only when the server sends `OTP_REQUIRED`,
7. send `OTP|<code>`,
8. display the final authentication result,
9. close cleanly.

## Use `getpass` for Password Input

Instead of:

```python
password = input("Password: ")
```

use:

```python
from getpass import getpass

password = getpass("Password: ")
```

`getpass()` prevents the password from being visibly echoed on the terminal while the user types.

### Important

This protects the password from **shoulder surfing on the local screen**. It does **not** encrypt the password while it travels across the network.

In [ ]:
from getpass import getpass

username = input("Username: ")
password = getpass("Password: ")

print("Username:", username)
print("Password was collected without echoing it.")

# Part 12 — Integrate and Test the Complete System

Run the server in one terminal:

```bash
python auth_server.py
```

Run the client in another:

```bash
python auth_client.py
```

Demonstrate the complete successful workflow:

```text
Username → Password → OTP → ACCESS_GRANTED
```

Then test failure cases one at a time.

Your goal is not simply to make successful login work. A security application must also behave correctly when authentication fails.

# Part 13 — Security and Robustness Tests

Demonstrate at least the following:

### Test 1 — Correct Password + Correct OTP

Expected:

```text
ACCESS_GRANTED
```

### Test 2 — Incorrect Password

The server must reject authentication and must **not** proceed to the OTP stage.

### Test 3 — Incorrect OTP

A correct password followed by an incorrect OTP must not grant access.

### Test 4 — Expired OTP

Generate an OTP, allow its time window to expire, and then test it.

### Test 5 — Invalid Username

The server should respond gracefully and should not crash.

### Test 6 — OTP Before Password Verification

Try sending an OTP without completing the password stage.

The server should reject the request.

### Test 7 — Malformed Message

For example:

```text
AUTH|alice
```

or another incomplete protocol message.

The server should handle it gracefully.

### Test 8 — Repeated Failed Attempts

Demonstrate several failed authentication attempts and observe the server behavior.

For the core lab, you are not required to build a full account-lockout system, but you should discuss why unlimited attempts are a security concern.

# Part 14 — Investigate Clock Synchronization

TOTP depends on time.

If the client/user device and server have significantly different clocks, the OTP generated by one side may not match what the server expects.

### Investigation

Answer:

1. Why does TOTP depend on synchronized time?
2. What happens when a code is submitted after its validity window?
3. Why might a small amount of clock tolerance sometimes be necessary?
4. What security trade-off could occur if the server accepts too many neighboring time windows?

You do not need to modify your computer's system clock for this investigation.

# Part 15 — Security Reflection

Answer the following questions:

1. Why should passwords not be stored in plaintext?
2. What is the purpose of hashing a password before storage?
3. Why is plain SHA-256 alone insufficient for production password storage?
4. What additional protection does TOTP provide beyond a password?
5. How can TOTP reduce the usefulness of a captured one-time code after it expires?
6. What happens when the user and server clocks are significantly out of sync?
7. What happens if an attacker obtains the user's TOTP secret?
8. Why should repeated failed login attempts be limited or monitored?
9. Does `getpass()` protect the password while it travels across the network? Explain.
10. What additional security would be required before deploying this authentication system in the real world?

# Part 16 — Security Analysis

Write a **300–400 word security analysis** of your authentication system.

Your analysis must reference your own implementation and address:

- how passwords are stored and verified,
- how TOTP changes the authentication workflow,
- threats that password + TOTP helps mitigate,
- remaining attack vectors,
- the security implications of transmitting credentials over a raw TCP connection,
- the limitations of using SHA-256 directly for password storage,
- the risk of TOTP secret compromise,
- the effect of repeated authentication attempts,
- improvements you would make for a real-world deployment.

Do not simply define hashing and OTP. Analyze the security properties and limitations of the system you actually built.

# Part 17 —  Challenge: Add Failed-Login Protection

As the final part of your authentication system, implement protection against repeated failed login attempts. Your server must track failed authentication attempts and restrict further authentication after three consecutive failures.Add a simple failed-login counter.

For example:

```text
Maximum failed attempts: 3

Attempt 1 → ACCESS_DENIED
Attempt 2 → ACCESS_DENIED
Attempt 3 → ACCESS_DENIED
            Too many failed attempts.
            Authentication blocked.
```

**Failed Attempt Counting**: The server must maintain a counter for consecutive failed authentication attempts. An incorrect password or an incorrect/expired OTP counts as one failed attempt. After three consecutive failed attempts, the server must deny further authentication attempts for that connection and display an appropriate message. A successful authentication resets the failed-attempt counter.

### Challenge Questions

1. Where should the failed-attempt counter be stored?
2. Should password failures and OTP failures count the same way?
3. Why can permanent account lockout itself create a denial-of-service problem?
4. How would a production system improve on this simple approach?

Keep the challenge focused on defensive authentication behavior.

# Deliverables

Submit:

- `auth_server.py`
- `auth_client.py`
- a short **MP4 screen recording** demonstrating:
  - successful password + OTP authentication,
  - incorrect password,
  - incorrect or expired OTP,
  - invalid username,
  - OTP-before-password rejection,
  - malformed input handling,
  - repeated failed attempts,
- answers to the Security Reflection questions,
- a **300–400 word Security Analysis**.

If you complete the Challenge, include your modified code and Challenge answers.

Submit **one PDF** containing the reflection answers, security analysis, and cloud links to the required technical artifacts.

### Filename

```text
Lab3-FirstName-LastName-StudentNumber.pdf
```

## Before You Submit

Confirm that:

- passwords are not stored in plaintext,
- the server verifies the password before OTP,
- the same user-specific TOTP secret is used consistently,
- invalid authentication attempts do not crash the server,
- the client uses `getpass()` for password entry,
- both client and server close sockets cleanly,
- you understand that the raw TCP connection does not encrypt credentials,
- your analysis discusses both the strengths **and limitations** of your implementation.